In [ ]:
# Run from the repository root so that `utils` and the relative data paths resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
os.makedirs("figures", exist_ok=True)  # output folder of the figures (not tracked by git)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams['font.size'] = 18
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob
import json

In [ ]:
# Speed distribution by category

def main():
    # Load data
    speed_list = []


    complete_list = glob(
        "data/labels_json/*_xs_cat_vehicles.json"
    )

    for file_path in complete_list:
        # e.g. 2024-08-29_09-01-07_<site>_xs_cat_vehicles.json
        time_part = Path(file_path).stem.split("_")[1]
        hour = int(time_part.split("-")[0]) 
        if hour < 10 or hour >= 20:
            continue
        with open(file_path, "r") as file:
            # print(f"Processing file: {file_path}")
            data = json.load(file)["data"]
            speed_list += [(v["type"], v["speed"]) for v in data.values()]
    
    

    # Figure settings
    plt.figure(figsize=(6, 4))
    color = "#F8AC8C"
    # plt.rcParams['font.family'] = "Times New Roman"
    # plt.rcParams['font.size'] = 16

    # Separate data and group by category
    class_1_speeds = np.array([speed for cls, speed in speed_list if cls == 1])
    class_2_speeds = np.array([speed for cls, speed in speed_list if cls == 2])

    print("Class 1 (Car):")
    print("  Mean speed:", class_1_speeds.mean())
    print("  std:", class_1_speeds.std())

    print("Class 2 (Truck):")
    print("  Mean speed:", class_2_speeds.mean())
    print("  std:", class_2_speeds.std())

    # Use seaborn to plot KDE curves
    sns.kdeplot(class_1_speeds, color="#7da6c6", label="Light", fill=True)
    sns.kdeplot(class_2_speeds, color="#eaaa60", label="Heavy", fill=True)
    plt.xlim(25, 150)
    plt.yticks(np.arange(0, 0.06, 0.01))
    plt.legend(loc="upper right")

    # Add labels and title
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Probability density")
    # plt.title("Speed Distribution by Category")
    plt.legend()

   
    # Save as svg
    # plt.savefig("figures/speed_distribution_kde.svg", format="svg", bbox_inches="tight")
    plt.savefig("figures/speed_pdf.pdf", dpi=300, bbox_inches='tight')
    
    # print(values)

main()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator


def plot_loss(fname="runs/loss.csv", threshold=3):
    df = pd.read_csv(fname)

    df["Train Loss Filtered"] = df["Train Loss"]
    df["Val Loss Filtered"] = df["Val Loss"]

    fig, ax = plt.subplots(figsize=(5, 4))
    
    sns.lineplot(data=df, x="Epoch", y="Train Loss Filtered", ax=ax, label="Train")
    sns.lineplot(data=df, x="Epoch", y="Val Loss Filtered", ax=ax, label="Val")

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_yscale("log")  # Set y-axis to logarithmic scale
    ax.set_ylim(1e-2, 1) 
    ax.set_xticks([0, 50, 100, 150])


    plt.tight_layout()
    plt.savefig("figures/loss.png", dpi=300)
    plt.show()

    # Save as pdf
    # fig.savefig("figures/loss.pdf", dpi=300, bbox_inches='tight')

plot_loss()


In [ ]:
def add_nan(arr, num_insert=10):
    # Get the dimensions of the array
    num_rows, num_cols = arr.shape
    # Calculate the new number of columns
    new_cols = num_cols + num_insert
    # Create a new array filled with NaN values
    arr_with_nan = np.full((num_rows, new_cols), np.nan)

    # Calculate the position to insert NaN values
    split_index = num_cols // 2  # Insert NaN in the middle
    # Copy the first half of the original array to the new array
    arr_with_nan[:, :split_index] = arr[:, :split_index]
    # Copy the second half of the original array to the new array
    arr_with_nan[:, split_index + num_insert :] = arr[:, split_index:]

    return arr_with_nan


# Plot prediction vs ground truth hourly traffic
def main(run, unit="hour", t=0, gt=None):

    run = Path(run) / "Test"

    if gt is None:
        gt = run

    # Define class labels
    cls_dict = {0: "Light, Left overtaking lane", 1: "Light, Right slow lane", 2: "Heavy, Left overtaking lane", 3: "Heavy, Right slow lane"}

    # Load prediction data
    run_folder = Path(run)
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path) for path in sorted(pred_path_list)]
    pred_arr = np.concatenate(pred_arr_list, axis=0)

    # Load ground truth data
    gt_folder = Path(gt)
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path) for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=0)

    # Select data to plot based on the unit (hour or minute)
    if unit == "hour":
        # Calculate hourly traffic for each class
        pred_hour = np.zeros((4, 80))
        gt_hour = np.zeros((4, 80))
        for i in range(len(pred_arr_list)):
            quarter = pred_arr_list[i].shape[0] // 4  # 180
            for j in range(4):
                pred_hour[:, 4*i + j] = np.sum(pred_arr_list[i][j * quarter : (j + 1) * quarter, :], axis=0)
                gt_hour[:, 4*i + j] = np.sum(gt_arr_list[i][j * quarter : (j + 1) * quarter, :], axis=0)
    
        pred_plot = add_nan(pred_hour)
        gt_plot = add_nan(gt_hour)  # (4,90)
        # print(pred_plot[:,-1], gt_plot[:,-1])
    elif unit == "minute":
        # Calculate minute-level traffic for each class
        pred_minute = pred_arr.reshape(-1, 12, 4).sum(axis=1).T
        gt_minute = gt_arr.reshape(-1, 12, 4).sum(axis=1).T
        pred_plot = pred_minute[:, t*60:t*60+60]
        gt_plot = gt_minute[:, t*60:t*60+60]

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # Create a 2x2 grid of subplots
    plot_classes = [0, 1, 2, 3]
    lines = []
    labels = []
    for i, ax in enumerate(axes.flat):  # Iterate over each subplot in the 2x2 grid
        plot_cls = plot_classes[i]
        
        # Plot curves and get Line2D objects
        # print(gt_plot[plot_cls].shape)
        gt_line, = ax.plot(gt_plot[plot_cls], label='Ground Truth', color="#eaaa60")
        pred_line, = ax.plot(pred_plot[plot_cls], label='Predicted', color="#0d95ce")
        
        # Collect legend items only in the first subplot to avoid repetition
        if i == 0:
            lines.extend([gt_line, pred_line])
            labels.extend(["Ground Truth (from CV Prediction)", "Prediction"])

        # ax.set_xlabel('Minutes' if unit == "minute" else 'Hour')
        # ax.set_ylabel('Traffic Count')
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        if unit == "hour":
            tick_positions = [0, 39, 49, 89]
            tick_labels = ["10 AM", "8 PM", "10 AM", "8 PM"]
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=50, ha="right") 
            ax.text(0.5, 0.1, '... ...', fontsize=20, ha='center', va='center', color='black', transform=ax.transAxes)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

            if i == 2 or i == 3:
                ax.set_ylim(None, 40)
            if i == 0 or i == 1:
                ax.set_ylim(100, 360)

        # ax.legend(loc='upper right')
        
        ax.text(
            0.96, 0.95, f'({cls_dict[plot_cls]})', color='black', fontsize=18,
            ha='right', va='top', transform=ax.transAxes, 
            # bbox=dict(facecolor='white', edgecolor='black', linewidth=0.5) 
        )

        # ax.axvline(x=11, color='black', linestyle=':', lw=1)
        # ax.axvline(x=16, color='black', linestyle=':', lw=1) 

    # Add a unified legend
    fig.legend(lines, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=2)
    # fig.legend(lines, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=2)
    if unit == "minute":
        fig.supxlabel('Minutes', y=0.05)
    fig.supylabel('Traffic Volume', x=0.02)

    plt.tight_layout()  # Adjust layout to avoid overlap
    plt.show()

    # Save the figure in pdf format
    fname = run_folder.name
    fig.savefig(f"figures/dual_{unit}.pdf", bbox_inches='tight', dpi=300)



# Call the main function to plot hourly traffic prediction vs ground truth
# main("runs/peak_count", "hour")
main("runs/dual", "hour")
# main("runs/dual", "minute", 8)
# main("runs/dual", "minute", 7)
# main("runs/noaxle","hour",  gt="runs/noaxle")

In [ ]:
def main():
    # Create some sample data
    x = np.arange(0, 61, 3)  # Discrete x values
    y1 = np.array([0, 10, 15, 20, 25, 30, 30, 35, 40, 45, 42, 38, 36, 34, 34, 30, 25, 15, 15, 5,5])  # Randomly generate y1 data
    y2 = np.array([30, 30, 25, 22, 20, 15, 15, 14, 13, 18, 20, 22, 30, 38, 45, 45, 50, 40, 40, 35,30])  # Randomly generate y2 data

    # Create plot
    plt.figure(figsize=(5, 4))

    # Plot two discrete lines
    plt.plot(x, y1, linestyle='--',marker='', label='Ground Truth', color='#e68b81')
    plt.plot(x, y2, linestyle='-',marker='', label='Prediction', color='#7da6c6')

    # Fill the area between y1 and y2
    plt.fill_between(x, y1, y2, where=(y1 > y2), interpolate=True, color='#e68b81', alpha=0.5)
    plt.fill_between(x, y1, y2, where=(y2 >= y1), interpolate=True, color='#7da6c6', alpha=0.5)

    # Fill the area below y down to 0
    # plt.fill_between(x, y1, 0, interpolate=True, color='#eaaa60', alpha=0.05)
    # plt.fill_between(x, y2, 0, interpolate=True, color='#7da6c6', alpha=0.05)

    plt.text(2, 18, 'B',  color='black', fontsize=22)
    plt.text(24, 25, 'D',  color='black', fontsize=22) 
    plt.text(48, 30, 'C',  color='black',   fontsize=22)
    plt.text(24, 5, 'A', color='black', fontsize=22) 

    # Add legend
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.ylim(0, 70)
    plt.xlim(0, 60)
    # plt.xticks(ticks=np.arange(0, 21, 2))

    plt.xlabel('Time')
    plt.ylabel('Traffic Volume')
    # plt.gca().yaxis.set_major_locator(MaxNLocator(nbins=5))
    plt.gca().set_xticks([])
    plt.gca().set_yticks([])

    # Save as pdf
    plt.savefig("figures/accuracy.pdf", format="pdf", bbox_inches="tight")

    # Show plot
    plt.show()

main()

In [ ]:
# Speed distribution by category
def main():
    # Load data
    speed_list = []
    df = pd.read_feather("data/label.feather")
    grouped = df.groupby("type")

    type_dict = {str(key): group["speed"].to_numpy() for key, group in grouped}
    overall_speed = df["speed"].to_numpy()

    print(f"Mean: {overall_speed.mean()}")
    print(f"Std: {overall_speed.std()}")


    sns.kdeplot(type_dict["1"], color="#7da6c6", label="Light", fill=True)
    sns.kdeplot(type_dict["2"], color="#eaaa60", label="Heavy", fill=True)
    plt.xlim(25, 150)
    plt.yticks(np.arange(0, 0.06, 0.01))

    # Add labels and title
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Probably Density")
    # plt.title("Speed Distribution by Category")
    plt.legend()

   
    # Save as svg
    # plt.savefig("figures/speed_distribution_kde.pdf", format="pdf", bbox_inches="tight")
    
    # print(values)

main()

In [ ]:
# Single signal
def main():
    import matplotlib.ticker as ticker
    data_truck = np.load("data/tsv4.npy")[4,100:500,-4:]
    data_size = len(data_truck)
    print(data_size)

    # Plot
    plt.figure(figsize=(20,3))
    # plt.title("Acclecration Sensor Matlab")
    
    x = [i/100 for i in range(data_size)]
    # plt.subplot(2, 1, 1) 
    # plt.ylim(-0.02, 0.1)
    # # plt.yticks([])
    # plt.plot(x,data_car,color="#5679ba",linewidth=3,label="Car")
    # plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    # plt.legend()

    plt.ylim(-0.3, 0.8)
    # plt.yticks([])
    # plt.plot(x,data_truck[:,3],color="#8a233f",linewidth=4,label="Truck")
    plt.plot(x,data_truck[:,1],color="#6D2F20",linewidth=4)
    plt.plot(x,data_truck[:,2],color="#B75347",linewidth=4)
    plt.plot(x,data_truck[:,3],color="#DF7E66",linewidth=4)
    # plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))


    plt.xticks([])
    plt.yticks([])
    # plt.ylabel("   ")
    # plt.text(-0.6, 1.1, "Normalized Strain", va='center', rotation='vertical')
    # plt.legend()
    # plt.show()
    plt.savefig("figures/strain_window.svg", format="svg", bbox_inches="tight")

main()

In [ ]:
# actual signal
# tsv4.npy stacks (along axis 0) the following signals:

# low_freq     - low-frequency wavelet component of the acceleration (0-7.8 Hz)
# mid_freq     - mid-frequency component (7.8-31.25 Hz)
# high_freq    - high-frequency component (31.25-62.5 Hz)
# acc_complete - raw acceleration (resampled from 250 Hz to 100 Hz)
# strain_data  - strain gauges (native 100 Hz)
def main():
    import matplotlib.ticker as ticker
    print(np.load("data/tsv4.npy").shape)
    # data_truck = np.load("data/tsv4.npy")[2,100:1500,-4:]
    data_truck =np.load("data/tsv4.npy")[4,100:1500,-4:]
    data_size = len(data_truck)
    print(data_size)

    # Plot
    plt.figure(figsize=(4,3))
    # plt.title("Acclecration Sensor Matlab")
    
    x = [i/100 for i in range(data_size)]
    # plt.subplot(2, 1, 1) 
    # plt.ylim(-0.02, 0.1)
    # # plt.yticks([])
    # plt.plot(x,data_car,color="#5679ba",linewidth=3,label="Car")
    # plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    # plt.legend()

    plt.ylim(-0.8, 0.8)
    # plt.yticks([])
    # plt.plot(x,data_truck[:,3],color="#8a233f",linewidth=4,label="Truck")
    plt.plot(x,data_truck[:,1],color="#6D2F20",linewidth=1)
    plt.plot(x,data_truck[:,2],color="#B75347",linewidth=1)
    plt.plot(x,data_truck[:,3],color="#DF7E66",linewidth=1)
    # plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))


    plt.xticks([])
    plt.yticks([])
    # plt.ylabel("   ")
    # plt.text(-0.6, 1.1, "Normalized Strain", va='center', rotation='vertical')
    # plt.legend()
    # plt.show()
    # plt.savefig("figures/actual.svg", format="svg", bbox_inches="tight")

main()

In [ ]:
def add_nan(arr, num_insert=10):
    # Get the dimensions of the array
    num_rows, num_cols = arr.shape
    # Calculate the new number of columns
    new_cols = num_cols + num_insert
    # Create a new array filled with NaN values
    arr_with_nan = np.full((num_rows, new_cols), np.nan)

    # Calculate the position to insert NaN values
    split_index = num_cols // 2  # Insert NaN in the middle
    # Copy the first half of the original array to the new array
    arr_with_nan[:, :split_index] = arr[:, :split_index]
    # Copy the second half of the original array to the new array
    arr_with_nan[:, split_index + num_insert :] = arr[:, split_index:]

    return arr_with_nan


# Graph abstract result
def main(run, unit="hour", t=0, gt=None):
    run = Path(run) / "Test"
    if gt is None:
        gt = run

    # Define class labels
    cls_dict = {0: "Light, Overtaking lane", 1: "Light, Slow lane", 2: "Heavy, Overtaking lane", 3: "Heavy, Slow lane"}

    # Load prediction data
    run_folder = Path(run)
    pred_path_list = glob(str(run_folder / "*_norm.npy"))
    pred_arr_list = [np.load(path) for path in sorted(pred_path_list)]
    pred_arr = np.concatenate(pred_arr_list, axis=0)

    # Load ground truth data
    gt_folder = Path(gt)
    gt_path_list = glob(str(gt_folder / "*_gt.npy"))
    gt_arr_list = [np.load(path) for path in sorted(gt_path_list)]
    gt_arr = np.concatenate(gt_arr_list, axis=0)

    # Select data to plot based on the unit (hour or minute)
    if unit == "hour":
        # Calculate hourly traffic for each class
        pred_hour = np.zeros((4, 80))
        gt_hour = np.zeros((4, 80))
        for i in range(len(pred_arr_list)):
            quarter = pred_arr_list[i].shape[0] // 4  # 180
            for j in range(4):
                pred_hour[:, 4*i + j] = np.sum(pred_arr_list[i][j * quarter : (j + 1) * quarter, :], axis=0)
                gt_hour[:, 4*i + j] = np.sum(gt_arr_list[i][j * quarter : (j + 1) * quarter, :], axis=0)
    
        pred_plot = add_nan(pred_hour)
        gt_plot = add_nan(gt_hour)  # (4,90)
        # print(pred_plot[:,-1], gt_plot[:,-1])
    elif unit == "minute":
        # Calculate minute-level traffic for each class
        pred_minute = pred_arr.reshape(-1, 12, 4).sum(axis=1).T
        gt_minute = gt_arr.reshape(-1, 12, 4).sum(axis=1).T
        pred_plot = pred_minute[:, t*60:t*60+60]
        gt_plot = gt_minute[:, t*60:t*60+60]

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))  # Create a 2x2 grid of subplots
    plot_classes = [0, 1, 3]
    lines = []
    labels = []
    for i, ax in enumerate(axes.flat):  # Iterate over each subplot in the 2x2 grid
        plot_cls = plot_classes[i]
        
        # Plot curves and get Line2D objects
        # print(gt_plot[plot_cls].shape)
        gt_line, = ax.plot(gt_plot[plot_cls], label='Ground Truth', color="#eaaa60")
        pred_line, = ax.plot(pred_plot[plot_cls], label='Predicted', color="#0d95ce")
        
        # Collect legend items only in the first subplot to avoid repetition
        if i == 0:
            lines.extend([gt_line, pred_line])
            labels.extend(["Ground Truth", "Prediction"])

        # ax.set_xlabel('Minutes' if unit == "minute" else 'Hour')
        # ax.set_ylabel('Traffic Count')
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        if unit == "hour":
            tick_positions = [0, 39, 49, 89]
            tick_labels = ["10 AM", "8 PM", "10 AM", "8 PM"]
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=50, ha="right") 
            ax.text(0.5, 0.1, '... ...', fontsize=20, ha='center', va='center', color='black', transform=ax.transAxes)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

            if i == 2 or i == 3:
                ax.set_ylim(None, 40)
            if i == 0 or i == 1:
                ax.set_ylim(100, 360)

        # ax.legend(loc='upper right')
        
        ax.text(
            0.96, 0.95, f'({cls_dict[plot_cls]})', color='black', fontsize=18,
            ha='right', va='top', transform=ax.transAxes, 
            # bbox=dict(facecolor='white', edgecolor='black', linewidth=0.5) 
        )

        # ax.axvline(x=11, color='black', linestyle=':', lw=1)
        # ax.axvline(x=16, color='black', linestyle=':', lw=1) 

    # Add a unified legend
    fig.legend(lines, labels, loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=2)
    # fig.legend(lines, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=2)
    if unit == "minute":
        fig.supxlabel('Minutes', y=0.05)
    fig.supylabel('Traffic Volume', x=0.04, y=0.6, weight="bold")

    plt.tight_layout()  # Adjust layout to avoid overlap
    plt.show()

    # Save the figure in pdf format
    fname = run_folder.name
    fig.savefig(f"figures/draft.svg", bbox_inches='tight', dpi=300)



plt.rcParams['font.weight'] = 'bold'
# Call the main function to plot hourly traffic prediction vs ground truth
# main("runs/peak_count", "hour")
main("runs/dual", "hour")
# main("runs/dual", "minute", 8)
# main("runs/dual", "minute", 7)
# main("runs/noaxle","hour",  gt="runs/noaxle")

In [ ]:
# Actual signal plot
import pandas as pd
import numpy as np
from glob import glob
import matplotlib.pyplot as plt

# --- Config ---
plot_start_time = pd.Timestamp("2024-08-22 12:41:16.010000")
plot_duration = 15  # seconds

acc_sampling_rate = 250  # Hz
strain_sampling_rate = 100  # Hz

acc_dir = "dataset/acceleration/*.pickle"
strain_file = glob("dataset/strain/*_strain_20240822121113_20240822185134.pickle")[0]

# --- Load acceleration ---
acc_file_list = glob(acc_dir)
acc_list = []
for f in acc_file_list:
    acc_df = pd.read_pickle(f)['accelerations']
    acc_data = acc_df.iloc[:, [6, 4, 2, 0]]  # VDC6-7-8-9-10-5-4-3-2-1
    acc_list.append(acc_data)

acc_complete = pd.concat(acc_list, axis=0)
acc_index = acc_complete.index
acc_complete = np.expand_dims(acc_complete.to_numpy(), axis=0)
print("acc_complete shape:", acc_complete.shape)

# --- Load strain ---
strain_df = pd.read_pickle(strain_file)
strain_data = strain_df.iloc[:, [3, 9, 8, 0]]  # last 4 channels of the original 10-channel order
print(f"Strain data start from {strain_data.index[0]} to {strain_data.index[-1]}")

# --- Locate the start of the requested window ---
# use searchsorted (not get_indexer) because the timestamp index has duplicate values
acc_start_idx = acc_index.searchsorted(plot_start_time)
acc_end_idx = acc_start_idx + plot_duration * acc_sampling_rate

strain_start_idx = strain_data.index.searchsorted(plot_start_time)
strain_end_idx = strain_start_idx + plot_duration * strain_sampling_rate

# --- Slice last 4 channels ---
acc_plot_data = acc_complete[0, acc_start_idx:acc_end_idx, -4:]
strain_plot_data = strain_data.iloc[strain_start_idx:strain_end_idx, -4:].to_numpy()

t_acc = np.arange(acc_plot_data.shape[0]) / acc_sampling_rate
t_strain = np.arange(strain_plot_data.shape[0]) / strain_sampling_rate

# --- Plot: two stacked subplots ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top: acceleration
for i in range(acc_plot_data.shape[1]):
    axes[0].plot(t_acc, acc_plot_data[:, i], label=f"A{i+5}")
axes[0].set_ylabel("Acceleration (g)")
# axes[0].set_title("Raw acceleration signals")
axes[0].legend(loc="upper right", ncol=4)
axes[0].grid(True)

# Bottom: strain
for i in range(strain_plot_data.shape[1]):
    axes[1].plot(t_strain, strain_plot_data[:, i], label=f"S{i+5}")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Strain (µm/m)")
# axes[1].set_title("Raw strain signals")
axes[1].legend(loc="upper right", ncol=4)
axes[1].grid(True)

axes[1].set_xlim(0, plot_duration)
fig.tight_layout()
plt.savefig("figures/actual_signals.pdf", format="pdf", bbox_inches="tight")
plt.show()
